# Siguiente token, frecuencia y contexto — solución comentada

¿Qué convierte una lista de números en información? En texto importa qué token
puede seguir; en audio, qué frecuencias están presentes. Los datos sintéticos de
este cuaderno (CC0-1.0) permiten inspeccionar el mecanismo completo en CPU, sin
red ni pesos externos. Un modelo de bigramas no es un Transformer y una FFT no
es un encoder preentrenado: son modelos pequeños que dejan ver la idea.


In [ ]:
from pathlib import Path
import json
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.nn import functional as F

torch.set_num_threads(1)
torch.manual_seed(23)
rng = np.random.default_rng(23)
plt.rcParams.update({'figure.figsize': (10, 3), 'axes.spines.top': False,
                     'axes.spines.right': False, 'font.size': 10})


In [ ]:
alpha = 0.1  # suavizado: nunca asignar probabilidad cero


## Qué tan sorprendente resulta el siguiente token

Cada fila de la figura es una distribución de probabilidad. La pérdida mide
sorpresa; su exponencial es perplejidad. El vocabulario se obtiene solo de train.
Las frases de transferencia intercambian papeles y conservan palabras: permiten
ver qué estructura aprendió realmente este modelo.


In [ ]:
# Un corpus mínimo permite ver todas las probabilidades, sin tokenizer opaco.
corpus_train = [
    'el gato mira la luna', 'el gato mira el sol', 'el perro mira la luna',
    'el perro busca la pelota', 'la niña busca el gato', 'la niña mira el sol',
    'el niño busca la pelota', 'el niño mira la luna', 'la gata busca el sol',
] * 8
corpus_val = ['la niña mira la luna', 'el gato busca la pelota', 'el perro mira el sol']
corpus_transfer = ['la luna mira el gato', 'la pelota busca el niño', 'el sol busca la niña']
vocabulario = sorted(set(' '.join(corpus_train).split()) | {'<fin>', '<desconocido>'})
a_indice = {w: i for i, w in enumerate(vocabulario)}
contextos = ['<inicio>'] + vocabulario
contexto_a_indice = {w: i for i, w in enumerate(contextos)}
conteos = np.zeros((len(contextos), len(vocabulario)), dtype=float)

def pares(frase):
    tokens = [w if w in a_indice else '<desconocido>' for w in frase.split()]
    return zip(['<inicio>'] + tokens, tokens + ['<fin>'])
for frase in corpus_train:
    for anterior, siguiente in pares(frase):
        conteos[contexto_a_indice[anterior], a_indice[siguiente]] += 1
probas = (conteos + alpha) / (conteos.sum(axis=1, keepdims=True) + alpha * len(vocabulario))
assert np.allclose(probas.sum(axis=1), 1)

def perplejidad(frases, tabla):
    sorpresas = [-np.log(tabla[contexto_a_indice[a], a_indice[b]])
                 for frase in frases for a, b in pares(frase)]
    return float(np.exp(np.mean(sorpresas)))

uniforme = np.full_like(probas, 1 / len(vocabulario))
base_ppl = perplejidad(corpus_val, uniforme)
val_ppl = perplejidad(corpus_val, probas)
transfer_ppl = perplejidad(corpus_transfer, probas)
assert np.isclose(base_ppl, len(vocabulario))
# Caso manual independiente: las cinco probabilidades observadas son
# 1/4, 1/2, 1/8, 1/4, 1/2. Su producto es 2**-9 y la PPL es 2**(9/5).
manual = np.full_like(probas, 1 / len(vocabulario))
for contexto, siguiente, probabilidad in [('<inicio>', 'el', .25), ('gato', '<fin>', .125)]:
    fila = manual[contexto_a_indice[contexto]]
    fila[:] = (1 - probabilidad) / (len(vocabulario) - 1)
    fila[a_indice[siguiente]] = probabilidad
manual[contexto_a_indice['el']] = 0
manual[contexto_a_indice['el'], a_indice['gato']] = .5
manual[contexto_a_indice['el'], a_indice['<fin>']] = .5
assert np.allclose(manual.sum(axis=1), 1)
assert np.isclose(perplejidad(['el gato', 'el'], manual), 2 ** (9 / 5))
if alpha in (.1, .5):
    assert val_ppl < base_ppl, 'El caso de referencia no supera la distribución uniforme'
fig, ax = plt.subplots(figsize=(10, 6))
im = ax.imshow(probas, cmap='magma', aspect='auto')
ax.set_xticks(range(len(vocabulario)), vocabulario, rotation=65, ha='right')
ax.set_yticks(range(len(contextos)), contextos)
ax.set(xlabel='Siguiente token', ylabel='Token anterior', title='Qué espera el modelo')
fig.colorbar(im, ax=ax, label='Probabilidad'); fig.tight_layout(); plt.show()
plt.bar(['Uniforme', 'Validación', 'Frases invertidas'], [base_ppl, val_ppl, transfer_ppl])
plt.ylabel('Perplejidad (menor es mejor)'); plt.show()
print({'uniforme': round(base_ppl, 3), 'validación': round(val_ppl, 3),
       'transferencia': round(transfer_ppl, 3)})


Un bigrama solo recuerda un token. Una Transformer causal puede usar un
contexto mayor, pero debe ocultar el futuro. El score de siguiente token no mide
por sí solo veracidad ni coherencia: mira qué frases invertidas reciben una
probabilidad alta por compartir transiciones locales.


## La misma señal, otra representación

Las dos clases tienen frecuencias distintas y fases aleatorias. Sus ondas pueden
parecer diferentes aun dentro de la misma clase; el espectro hace visible lo que
permanece. Cada grabación entera pertenece a un único split, sin ventanas
solapadas compartidas. El remuestreo conserva duración y cambia el número de
muestras; etiquetar la tasa de otra manera no hace esa transformación.


In [ ]:
from scipy.signal import resample_poly
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

fs = 4000

def grabaciones(n, semilla, ruido=.3):
    r = np.random.default_rng(semilla)
    etiquetas = np.arange(n) % 2
    r.shuffle(etiquetas)
    tiempo = np.arange(fs) / fs
    ondas = []
    for y in etiquetas:
        frecuencia = 180 if y == 0 else 620
        onda = np.sin(2*np.pi*frecuencia*tiempo + r.uniform(0, 2*np.pi))
        ondas.append((onda + r.normal(0, ruido, fs)).astype(np.float32))
    return np.array(ondas), etiquetas

ondas_train, ya = grabaciones(80, 0)
ondas_val, yb = grabaciones(40, 1)
ondas_transfer, yc = grabaciones(40, 2, ruido=.9)
frecuencias = np.fft.rfftfreq(fs, d=1/fs)
fig, axes = plt.subplots(1, 2, figsize=(10, 3))
for i in [np.flatnonzero(ya == 0)[0], np.flatnonzero(ya == 1)[0]]:
    onda, y = ondas_train[i], ya[i]
    axes[0].plot(np.arange(100)/fs, onda[:100], alpha=.7, label=f'Clase {y}')
    axes[1].plot(frecuencias, np.abs(np.fft.rfft(onda))/fs, alpha=.7)
axes[0].set(xlabel='Segundos', ylabel='Amplitud'); axes[0].legend()
axes[1].set(xlabel='Hz', ylabel='Magnitud / n', xlim=(0, 1000)); plt.show()

# Remuestrear modifica los datos, no solo el número fs.
remuestreada = resample_poly(ondas_val[0], up=1, down=2)
fs_nueva = fs // 2
assert np.isclose(len(remuestreada)/fs_nueva, len(ondas_val[0])/fs)
f_nueva = np.fft.rfftfreq(len(remuestreada), d=1/fs_nueva)
pico = f_nueva[np.abs(np.fft.rfft(remuestreada))[1:].argmax()+1]
assert abs(pico - (180 if yb[0] == 0 else 620)) <= 1

# Se recogen features, etiquetas e IDs juntos incluso al barajar el recorrido.
def extraer(ondas, etiquetas, semilla):
    filas, y, ids = [], [], []
    for i in np.random.default_rng(semilla).permutation(len(ondas)):
        onda = ondas[i]
        espectro = np.abs(np.fft.rfft(onda)) / len(onda)
        filas.append([espectro[(frecuencias > 150) & (frecuencias < 210)].sum(),
                      espectro[(frecuencias > 590) & (frecuencias < 650)].sum()])
        y.append(etiquetas[i]); ids.append(i)
    return np.array(filas), np.array(y), np.array(ids)
fa, la, ida = extraer(ondas_train, ya, 8)
fb, lb, idb = extraer(ondas_val, yb, 9)
fc, lc, idc = extraer(ondas_transfer, yc, 10)
assert np.array_equal(lb, yb[idb])
clf = make_pipeline(StandardScaler(), LogisticRegression(random_state=0))
clf.fit(fa, la)
clase_mayoritaria = np.bincount(la).argmax()
base_audio = accuracy_score(lb, np.full(lb.shape, clase_mayoritaria))
val_audio = accuracy_score(lb, clf.predict(fb))
transfer_audio = accuracy_score(lc, clf.predict(fc))
assert np.isfinite(fa).all() and np.isfinite(fb).all() and np.isfinite(fc).all()
assert val_audio > base_audio + .3, 'Las features no separan las frecuencias del caso fijo'
assert transfer_audio > .85, 'La clasificación perdió robustez al ruido del caso fijo'
print({'accuracy clase mayoritaria': base_audio, 'accuracy espectral': val_audio,
       'accuracy con más ruido': transfer_audio})
fig, ax = plt.subplots(figsize=(5, 3))
ax.scatter(fb[:, 0], fb[:, 1], c=lb, cmap='coolwarm')
ax.set(xlabel='Magnitud de banda baja', ylabel='Magnitud de banda alta'); plt.show()


## Lo que el padding puede esconder

El promedio de estados debe excluir posiciones que no representan audio. Un
encoder puede submuestrear el tiempo: la longitud válida de su salida debe
calcularse con su arquitectura, no copiando la máscara de la onda.


In [ ]:
# Un encoder puede acortar el eje temporal. La máscara debe vivir en SU salida.
# Aquí los vectores son inventados para aislar el efecto del padding.
estados = torch.tensor([[[2., 4.], [4., 2.], [0., 0.]],
                        [[2., 4.], [4., 2.], [6., 6.]]])
longitudes_salida = torch.tensor([2, 3])
mascara = torch.arange(3)[None, :] < longitudes_salida[:, None]
pooling_ingenuo = estados.mean(dim=1)
pooling_valido = (estados * mascara[..., None]).sum(1) / longitudes_salida[:, None]
assert torch.allclose(pooling_valido[0], torch.tensor([3., 3.]))
print('Sin máscara:', pooling_ingenuo.numpy(), '\nCon máscara:', pooling_valido.numpy())


El suavizado evita sorpresa infinita en transiciones nuevas, pero demasiado
suavizado acerca todas las filas a una distribución uniforme. Más contexto no
siempre es mejor con pocos datos. En audio, separar las fases de las frecuencias
hace que un clasificador pequeño sea suficiente para este problema sintético.
Eso no demuestra que baste para voces o sonidos reales.

Crea dos clases con el mismo espectro global y diferente orden temporal. La FFT
promediada pierde esa diferencia: ahora una ventana, un espectrograma o un encoder
secuencial pueden aportar información que tu resumen descartó.


### Extensión con un encoder real, cuando estén preparados sus pesos

El material oficial de Find the Order incluye un ejemplo de
`facebook/wav2vec2-base-960h` y permite usar el encoder de Whisper. Usa el notebook
oficial para preparar modelo y procesador, remuestrea a su tasa esperada y recoge
IDs, etiquetas y vectores en el mismo recorrido. Las longitudes del pooling deben
corresponder a la salida del modelo. Compara contra la referencia espectral en el
mismo split por grabación.

Esta extensión no se ha ejecutado como parte de este cuaderno; requiere los
activos y el entorno del modelo. No sustituye pesos ausentes por aleatorios.
[Ejemplo oficial](https://github.com/IOAI-official/IOAI-2026/blob/main/Individual-Contest/1_Find_the_Order/code/baseline-original/solution.ipynb).


El archivo siguiente es un resumen técnico para verificar el cuaderno automáticamente; no hay un formulario que completar.


In [ ]:
resultado = {'laboratorio': '09_lenguaje_audio', 'version': 'solución comentada',
             'metrica': 'Perplejidad de siguiente token, menor es mejor', 'split': 'corpus train fijo; frases nuevas de validación; frases con papeles invertidos de transferencia',
             'baseline': base_ppl, 'validacion': val_ppl, 'transferencia': transfer_ppl}
assert all(np.isfinite(resultado[k]) for k in ('baseline', 'validacion', 'transferencia'))
Path('resultado.json').write_text(json.dumps(resultado, ensure_ascii=False, indent=2) + '\n')
